# DemoNotebook — API keys and data uploads

This notebook shows how **LibreQuant** loads credentials and how to use **market data** (`get_bars`) and **files you uploaded** under `data/uploads/` (`read_tabular`).

**Getting this file into Jupyter:** If you cloned the repo, copy or upload `librequant/notebooks/DemoNotebook.ipynb` into your notebook workspace (same tree as in the **Notebook library** in the app), then open it here.

## 1. How API keys reach the kernel

- Add keys in the LibreQuant app under **Data sources** (or edit `librequant/.env.local` on the machine running Next.js). Managed names include `ALPACA_API_KEY`, `ALPACA_SECRET_KEY`, and custom uppercase variables you define.
- The app can sync a file into this workspace: `config/credentials.env`. The `librequant.data` package loads it automatically on data calls.
- **`load_data_source_secrets()`** reloads that file into `os.environ` (useful after you change keys without restarting Docker).
- **`get_bars()`** calls `load_data_source_secrets()` for you, so you often only need the imports below.

In [ ]:
import os

from librequant.data import get_bars, load_data_source_secrets

# Optional: force reload of config/credentials.env into the kernel environment
load_data_source_secrets()

# Example: confirm Alpaca vars are visible (values are not printed)
print("ALPACA_API_KEY set:", bool(os.environ.get("ALPACA_API_KEY", "").strip()))
print("ALPACA_SECRET_KEY set:", bool(os.environ.get("ALPACA_SECRET_KEY", "").strip()))

## 2. OHLCV bars — no API key (yfinance)

The default source is **`yfinance`**. No API key is required for a quick demo.

In [ ]:
df_yf = get_bars("AAPL", "2024-01-01", "2024-06-01", source="yfinance", interval="1d")
df_yf.head()

## 3. OHLCV bars — Alpaca (requires keys)

Set **Alpaca** keys in **Data sources**, save, then run the cell below. If keys are missing, you will get a clear error asking you to configure the environment.

In [ ]:
try:
    df_alp = get_bars("AAPL", "2024-01-01", "2024-06-01", source="alpaca", interval="1d")
    display(df_alp.head())
except Exception as e:
    print(type(e).__name__, ":", e)

## 4. Cached data

Repeated requests for the same symbol, range, source, and interval are served from Parquet under **`data/cache/ohlcv/`** in your workspace (no extra download when the cache covers the range).

In [ ]:
from librequant.data.paths import get_data_root

cache_root = get_data_root() / "cache" / "ohlcv"
print("Cache directory:", cache_root)
print("Parquet files (sample):", sorted(cache_root.glob("*.parquet"))[:5])

## 5. Uploaded files — `data/uploads/`

Use the app **Data sources → Data library** to upload CSV / Excel files. They land under **`data/uploads/`** (and subfolders you create).

- **`resolve_data_path("your_file.csv")`** returns the absolute path under `data/uploads/`.
- **`read_tabular(...)`** loads `.csv` or `.xlsx` into a pandas `DataFrame`.

In [ ]:
from pathlib import Path

from librequant.data import read_tabular, resolve_data_path
from librequant.data.paths import get_data_root

uploads = get_data_root() / "uploads"
print("Uploads folder:", uploads)

csv_files = sorted(uploads.glob("**/*.csv"))
xlsx_files = sorted(uploads.glob("**/*.xlsx"))
print("CSV files found:", len(csv_files))
for p in csv_files[:10]:
    print(" -", p.relative_to(uploads))

if csv_files:
    # Load relative to data/uploads (see resolve_data_path)
    rel = csv_files[0].relative_to(uploads)
    df = read_tabular(str(rel))
    display(df.head())
else:
    print("No CSV yet — upload one from Data sources → Data library, then re-run this cell.")

### Optional: resolve a path by name

If your file is `data/uploads/reports/sales.csv`, you can pass `reports/sales.csv` to `read_tabular` (or build the path with `resolve_data_path`).

In [ ]:
# Example (uncomment and fix the relative path after you upload a file):
# df = read_tabular("my_sheet.xlsx")
# df.head()

p = resolve_data_path("example_placeholder.csv")  # shows where LibreQuant would look
print("Resolved path would be:", p)